In [ ]:
%%capture
%pip install QuantumRingsLib numpy matplotlib


In [ ]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#

import os

my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "amber_quantum_rings"
precision = "single"

In [ ]:
from platform import python_version
print(python_version())
import os
import sys
import platform
from os import listdir
from os.path import isfile, join
from colorama import Fore, Style, init
from itertools import islice
from collections import namedtuple

if platform.system() == "Windows":
    cuda_path = os.getenv("CUDA_PATH", "")
    if "" == cuda_path:
        #set a hard-coded path
        cuda_path = "C:\\Program Files\\NVIDIA GPU Computing Toolkit\\CUDA\\v13.0\\bin\\x64"
    else:
        #create from the environment
        if "13" in cuda_path:
            cuda_path += "\\bin\\x64"
        else:
            cuda_path += "\\bin"
            
    os.add_dll_directory(cuda_path)


import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from QuantumRingsLib import OptimizeQuantumCircuit
from matplotlib import pyplot as plt
import numpy as np
import math
import time
from collections import Counter
import json
import csv
from enum import Enum

provider = QuantumRingsProvider(token=my_token, name=my_name)
backend = provider.get_backend(my_backend, precision = precision)
print("Account Name: ", provider.active_account()["name"], "\nMax Qubits: ", provider.active_account()["max_qubits"])

In [ ]:
# Build a Quantum Circuit

qc = QuantumCircuit(5, 5)

qc.h(0)
qc.x(1)

qc.id(3)
qc.t(4)
qc.s(0)
qc.tdg(1)
qc.sdg(2)
qc.sx(3)

qc.sxdg(4)
qc.y(0)
qc.z(1)
qc.p(math.pi/2, 2)
qc.r(math.pi/2, -math.pi/4, 3)

qc.rv(math.pi/2, -math.pi/2, -math.pi/4, 4)
qc.rx(-math.pi/4, 0)
qc.ry(math.pi/2, 1)
qc.rx(-math.pi, 2)
qc.u(math.pi/2, -math.pi/2, -math.pi/4, 3)

qc.u0(-math.pi/4, 4)
qc.u1(math.pi/2, 0)

qc.u2(-math.pi/2, -math.pi/2, 2)

qc.u3(math.pi/2, -math.pi/2, -math.pi/4, 3)

qc.delay(10, 4)

qc.cx(0, 1)
qc.cy(1, 2)
qc.cz(2, 3)
qc.ch(3, 4)
qc.cnot(0, 1)
qc.cp(math.pi/2, 1, 2)

qc.cs(2, 3)
qc.csdg(3, 4)
qc.csx(0, 1)
qc.crx(math.pi, 2, 3)
qc.cry(-math.pi/2, 3, 4)
qc.crz(math.pi, 0, 1)

qc.cu(math.pi/2, -math.pi/2, -math.pi/4, -math.pi/2, 2, 3)
qc.dcx(3, 4)
qc.ecr(0, 1)
qc.iswap(2, 3)
qc.rxx(-math.pi/2, 3, 4)
qc.ryy(math.pi/2, 0, 1)
qc.rzx(-math.pi/4, 1, 2)
qc.rzz(math.pi/4, 2, 3)
qc.swap(3, 4)

qc.cu1(-math.pi/2, 0, 1)

qc.cu3(math.pi/2, -math.pi/2, -math.pi/4, 1, 2)

# Take the inverse of this circuit.
qc_inverse = qc.inverse()


In [ ]:
# to find the mirror-fidelity or to find the peak size of the simulation, one shot is enough
# NOTE: If the circuit has resets or mid-circuit measurements, this mirror-fidelity does not work.
number_of_shots = 1

# start with a threshold of 4. Increase until a desired fidelity is achived. 
threshold = 4

In [ ]:

#Attach the inverse to the given circuit
qc.append(qc_inverse)

job = backend.run(
        qc, 
        shots=number_of_shots, 
        mode="async", 
        quiet=True, 
        performance="custom", 
        threshold=threshold
        )
job_monitor(job, quiet=True)
result = job.result()
print(f"Fidelity: {result.get_fidelity()} Peak Simulation Size: {result.get_peakmemorysize(0)}")
